In [ ]:
import subprocess

result = subprocess.run(
    ["ip", "netns", "list"],
    capture_output=True,
    text=True,
    timeout=5,
    check=True,
)

print(result.stdout or "No named network namespaces found.")

In [ ]:
result = subprocess.run(
    ["ip", "netns", "exec", "web01", "nft", "--version"],
    capture_output=True,
    text=True,
    timeout=5,
    check=True,
)

print(result.stdout.strip())

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from collections import Counter

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, clear_output, JSON, Markdown
from pydantic import BaseModel, ConfigDict, Field, ValidationError

NAMESPACE = "web01"
COMMAND_TIMEOUT = 15

In [ ]:
def collect_ruleset(namespace=NAMESPACE):
    command = [
        "ip", "netns", "exec", namespace,
        "nft", "-j", "list", "ruleset",
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=COMMAND_TIMEOUT,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Ruleset collection failed (exit {result.returncode}).\n"
            f"{result.stderr.strip()}"
        )

    try:
        payload = json.loads(result.stdout)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"nft returned invalid JSON: {exc}"
        ) from exc

    report = validate_ruleset(payload)

    if report["errors"]:
        raise ValueError(
            "Invalid ruleset structure:\n- "
            + "\n- ".join(report["errors"])
        )

    return {
        "namespace": namespace,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "ruleset": payload,
        "validation": report,
    }


def validate_ruleset(payload):
    report = {
        "valid": False,
        "errors": [],
        "warnings": [],
        "object_counts": {},
    }

    if not isinstance(payload, dict):
        report["errors"].append(
            "Top-level JSON value must be an object."
        )
        return report

    objects = payload.get("nftables")

    if not isinstance(objects, list):
        report["errors"].append(
            "Missing or invalid 'nftables' list."
        )
        return report

    counts = Counter()
    known_types = {
        "metainfo", "table", "chain", "rule",
        "set", "map", "element", "counter",
        "quota", "ct helper", "flowtable",
        "limit", "secmark", "synproxy",
    }

    for index, entry in enumerate(objects):
        if not isinstance(entry, dict) or len(entry) != 1:
            report["errors"].append(
                f"Entry {index}: expected an object with one type key."
            )
            continue

        kind, value = next(iter(entry.items()))
        counts[kind] += 1

        if not isinstance(value, dict):
            report["errors"].append(
                f"Entry {index} ({kind}): expected an object value."
            )
            continue

        if kind not in known_types:
            report["warnings"].append(
                f"Entry {index}: unfamiliar object type {kind!r}."
            )

        required = {
            "table": ("family", "name"),
            "chain": ("family", "table", "name"),
            "rule": ("family", "table", "chain", "expr"),
        }.get(kind, ())

        for field in required:
            if field not in value:
                report["errors"].append(
                    f"Entry {index} ({kind}): missing {field!r}."
                )

        if kind == "rule" and "expr" in value:
            if not isinstance(value["expr"], list):
                report["errors"].append(
                    f"Entry {index} (rule): 'expr' must be a list."
                )
            elif not all(
                isinstance(expr, dict) for expr in value["expr"]
            ):
                report["errors"].append(
                    f"Entry {index} (rule): each expression must be an object."
                )

    report["object_counts"] = dict(counts)
    report["valid"] = not report["errors"]
    return report

In [ ]:
snapshot = collect_ruleset()

raw_ruleset = snapshot["ruleset"]
objects = raw_ruleset["nftables"]

print(f"Namespace: {snapshot['namespace']}")
print(f"Collected: {snapshot['collected_at']}")
print(f"JSON objects: {len(objects)}")
print(f"Validation passed: {snapshot['validation']['valid']}")

display(pd.DataFrame([
    {"object_type": kind, "count": count}
    for kind, count in snapshot["validation"]["object_counts"].items()
]))

if snapshot["validation"]["warnings"]:
    display(Markdown("### Validation warnings"))
    for warning in snapshot["validation"]["warnings"]:
        print("•", warning)

In [ ]:
def objects_of(kind):
    return [
        entry[kind]
        for entry in objects
        if kind in entry
    ]


tables_df = pd.DataFrame([
    {
        "family": item.get("family"),
        "table": item.get("name"),
        "handle": item.get("handle"),
    }
    for item in objects_of("table")
])

chains_df = pd.DataFrame([
    {
        "family": item.get("family"),
        "table": item.get("table"),
        "chain": item.get("name"),
        "type": item.get("type"),
        "hook": item.get("hook"),
        "priority": item.get("prio"),
        "policy": item.get("policy"),
        "handle": item.get("handle"),
    }
    for item in objects_of("chain")
])

In [ ]:
def summarize_rule(rule):
    expressions = rule.get("expr", [])
    verdicts = []
    jumps = []
    gotos = []
    counter_values = []

    for expr in expressions:
        if not isinstance(expr, dict):
            continue

        for verdict in ("accept", "drop", "reject", "return"):
            if verdict in expr:
                verdicts.append(verdict)

        for key, destination in (("jump", jumps), ("goto", gotos)):
            if key in expr:
                value = expr[key]
                if isinstance(value, dict):
                    destination.append(value.get("target", "?"))
                else:
                    destination.append(str(value))

        if "counter" in expr:
            value = expr["counter"]
            if isinstance(value, dict):
                counter_values.append({
                    "packets": value.get("packets"),
                    "bytes": value.get("bytes"),
                })
            else:
                counter_values.append({
                    "packets": None,
                    "bytes": None,
                })

    known_packets = [
        c["packets"] for c in counter_values
        if isinstance(c["packets"], (int, float))
    ]
    known_bytes = [
        c["bytes"] for c in counter_values
        if isinstance(c["bytes"], (int, float))
    ]

    # Only report aggregate values if all counters have known values.
    packets = (
        sum(known_packets)
        if counter_values and len(known_packets) == len(counter_values)
        else None
    )
    byte_count = (
        sum(known_bytes)
        if counter_values and len(known_bytes) == len(counter_values)
        else None
    )

    return {
        "family": rule.get("family"),
        "table": rule.get("table"),
        "chain": rule.get("chain"),
        "handle": rule.get("handle"),
        "packets": packets,
        "bytes": byte_count,
        "counter_count": len(counter_values),
        "verdicts": ", ".join(verdicts),
        "jumps": ", ".join(jumps),
        "gotos": ", ".join(gotos),
        "expression_types": ", ".join(
            next(iter(expr), "?") for expr in expressions
        ),
        "expressions": json.dumps(expressions, sort_keys=True),
    }


rules_df = pd.DataFrame([
    summarize_rule(rule)
    for rule in objects_of("rule")
])

print(f"Tables: {len(tables_df)}")
print(f"Chains: {len(chains_df)}")
print(f"Rules: {len(rules_df)}")

In [ ]:
def options_for(series):
    values = sorted(series.dropna().astype(str).unique().tolist())
    return [("All", "__all__")] + [
        (value, value) for value in values
    ]


family_widget = widgets.Dropdown(
    options=options_for(rules_df["family"]),
    description="Family:",
    layout=widgets.Layout(width="320px"),
)

table_widget = widgets.Dropdown(
    options=options_for(rules_df["table"]),
    description="Table:",
    layout=widgets.Layout(width="320px"),
)

chain_widget = widgets.Dropdown(
    options=options_for(rules_df["chain"]),
    description="Chain:",
    layout=widgets.Layout(width="320px"),
)

search_widget = widgets.Text(
    description="Search:",
    placeholder="Expression, verdict, jump target...",
    layout=widgets.Layout(width="480px"),
)

rules_output = widgets.Output()


def render_rules(*_):
    df = rules_df.copy()

    if family_widget.value != "__all__":
        df = df[df["family"] == family_widget.value]

    if table_widget.value != "__all__":
        df = df[df["table"] == table_widget.value]

    if chain_widget.value != "__all__":
        df = df[df["chain"] == chain_widget.value]

    query = search_widget.value.strip()
    if query:
        mask = df[
            [
                "verdicts", "jumps", "gotos",
                "expression_types", "expressions",
            ]
        ].fillna("").astype(str).apply(
            lambda col: col.str.contains(
                query, case=False, regex=False
            )
        ).any(axis=1)

        df = df[mask]

    with rules_output:
        clear_output(wait=True)
        display(Markdown(f"**{len(df)} matching rules**"))

        visible_columns = [
            "family", "table", "chain", "handle",
            "packets", "bytes", "verdicts", "jumps",
            "gotos", "expression_types",
        ]

        display(
            df[visible_columns].reset_index(drop=True)
        )


for control in (
    family_widget, table_widget, chain_widget, search_widget
):
    control.observe(render_rules, names="value")

display(widgets.VBox([
    widgets.HBox([family_widget, table_widget]),
    widgets.HBox([chain_widget, search_widget]),
    rules_output,
]))

render_rules()

In [ ]:
def inspect_rule(family, table, chain, handle):
    matches = [
        rule for rule in objects_of("rule")
        if rule.get("family") == family
        and rule.get("table") == table
        and rule.get("chain") == chain
        and rule.get("handle") == handle
    ]

    if not matches:
        print("Rule not found in the current snapshot.")
        return

    display(JSON(matches[0]))


rule_picker = widgets.Dropdown(
    options=[
        (
            f"{r['family']} / {r['table']} / "
            f"{r['chain']} / handle {r['handle']}",
            i,
        )
        for i, r in enumerate(objects_of("rule"))
    ],
    description="Rule:",
    layout=widgets.Layout(width="700px"),
)

inspect_output = widgets.Output()


def show_selected_rule(change=None):
    with inspect_output:
        clear_output(wait=True)
        rule = objects_of("rule")[rule_picker.value]
        display(JSON(rule))


rule_picker.observe(show_selected_rule, names="value")
display(rule_picker, inspect_output)
show_selected_rule()

In [ ]:
def refresh_ruleset():
    global snapshot, raw_ruleset, objects
    global tables_df, chains_df, rules_df

    snapshot = collect_ruleset()
    raw_ruleset = snapshot["ruleset"]
    objects = raw_ruleset["nftables"]

    tables_df = pd.DataFrame([
        {
            "family": x.get("family"),
            "table": x.get("name"),
            "handle": x.get("handle"),
        }
        for x in objects_of("table")
    ])

    chains_df = pd.DataFrame([
        {
            "family": x.get("family"),
            "table": x.get("table"),
            "chain": x.get("name"),
            "type": x.get("type"),
            "hook": x.get("hook"),
            "priority": x.get("prio"),
            "policy": x.get("policy"),
            "handle": x.get("handle"),
        }
        for x in objects_of("chain")
    ])

    rules_df = pd.DataFrame([
        summarize_rule(rule)
        for rule in objects_of("rule")
    ])

    print("Refreshed:", snapshot["collected_at"])
    print("Rules:", len(rules_df))
    print("Validation:", snapshot["validation"]["valid"])

    render_rules()


refresh_button = widgets.Button(
    description="Refresh ruleset",
    icon="refresh",
    button_style="primary",
)

refresh_button.on_click(lambda _: refresh_ruleset())

display(refresh_button)

In [ ]:
from pathlib import Path

export_dir = Path("snapshots")
export_dir.mkdir(exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

export_path = export_dir / f"web01-{timestamp}.json"

export_path.write_text(
    json.dumps(snapshot, indent=2),
    encoding="utf-8",
)

print("Saved:", export_path)

In [ ]:
from collections import defaultdict
import json
import pandas as pd


def object_index(snapshot):
    """Index live nftables objects by their JSON object type."""
    result = defaultdict(list)

    for entry in snapshot["ruleset"]["nftables"]:
        if isinstance(entry, dict):
            for kind, value in entry.items():
                if isinstance(value, dict):
                    result[kind].append(value)

    return result


def object_id(obj):
    return (
        obj.get("family"),
        obj.get("table"),
        obj.get("name"),
    )


def walk_json(value):
    """Yield every nested dictionary and list item."""
    yield value

    if isinstance(value, dict):
        for child in value.values():
            yield from walk_json(child)

    elif isinstance(value, list):
        for child in value:
            yield from walk_json(child)


def count_elements(value):
    """
    Count encoded element entries conservatively.
    This is an inventory count, not a count of individual IPs
    represented by an interval.
    """
    if isinstance(value, list):
        return sum(count_elements(item) for item in value)

    if isinstance(value, dict):
        if "elem" in value:
            return 1

        return sum(count_elements(v) for v in value.values())

    return 0


def extract_map_references(rule):
    """
    Extract map/set references occurring within vmap expressions.
    Keep the expression JSON for manual verification.
    """
    references = []

    def walk(value, inside_vmap=False):
        if isinstance(value, dict):
            for key, child in value.items():
                in_vmap = inside_vmap or key == "vmap"

                # nft JSON commonly represents a map lookup
                # using a nested {"set": "map_name"} object.
                if in_vmap and key in {"set", "map"}:
                    if isinstance(child, str):
                        references.append(child.lstrip("@"))
                    elif isinstance(child, dict):
                        for nested in walk_json(child):
                            if isinstance(nested, dict):
                                name = nested.get("name")
                                if isinstance(name, str):
                                    references.append(name)

                walk(child, in_vmap)

        elif isinstance(value, list):
            for child in value:
                walk(child, inside_vmap)

    walk(rule.get("expr", []))

    return sorted(set(references))


idx = object_index(snapshot)

map_objects = idx["map"]
set_objects = idx["set"]
element_objects = idx["element"]
rule_objects = idx["rule"]

print(f"Maps: {len(map_objects)}")
print(f"Sets: {len(set_objects)}")
print(f"Element objects: {len(element_objects)}")
print(f"Rules: {len(rule_objects)}")

In [ ]:
elements_by_map = defaultdict(list)

for element in element_objects:
    elements_by_map[object_id(element)].append(element)

map_rows = []

for obj in map_objects + set_objects:
    identity = object_id(obj)
    map_rows.append({
        "family": obj.get("family"),
        "table": obj.get("table"),
        "name": obj.get("name"),
        "object_type": (
            "map" if obj in map_objects else "set"
        ),
        "key_type": obj.get("type"),
        "value_type": obj.get("map"),
        "flags": ", ".join(obj.get("flags", []))
            if isinstance(obj.get("flags"), list)
            else str(obj.get("flags", "")),
        "element_objects": len(elements_by_map[identity]),
        "handle": obj.get("handle"),
        "raw_json": json.dumps(obj, sort_keys=True),
    })

maps_df = pd.DataFrame(map_rows)

display(
    maps_df.sort_values(
        ["family", "table", "name"],
        na_position="last",
    ).reset_index(drop=True)
)

In [ ]:
reference_rows = []

for rule in rule_objects:
    refs = extract_map_references(rule)

    for ref_name in refs:
        reference_rows.append({
            "rule_family": rule.get("family"),
            "rule_table": rule.get("table"),
            "rule_chain": rule.get("chain"),
            "rule_handle": rule.get("handle"),
            "map_name": ref_name,
            "rule_expressions": json.dumps(
                rule.get("expr", []), sort_keys=True
            ),
        })

references_df = pd.DataFrame(reference_rows)

display(references_df)

In [ ]:
map_lookup = {
    object_id(obj): obj
    for obj in map_objects + set_objects
}

# Resolve references by name within the rule's family and table.
# If a name cannot be resolved, retain it as unresolved.
reference_audit = []

for row in reference_rows:
    candidates = [
        key for key in map_lookup
        if key[2] == row["map_name"]
        and key[0] == row["rule_family"]
        and key[1] == row["rule_table"]
    ]

    reference_audit.append({
        **row,
        "resolution": (
            "resolved" if len(candidates) == 1
            else "ambiguous" if len(candidates) > 1
            else "unresolved"
        ),
        "resolved_identity": (
            candidates[0] if len(candidates) == 1 else None
        ),
    })

reference_audit_df = pd.DataFrame(reference_audit)

display(reference_audit_df)

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Map/set nodes
for obj in map_objects + set_objects:
    family, table, name = object_id(obj)
    node_id = f"map:{family}:{table}:{name}"

    G.add_node(
        node_id,
        kind="map" if obj in map_objects else "set",
        label=f"{family} {table}\n{name}",
        family=family,
        table=table,
        name=name,
    )

# Rule nodes and rule-to-map edges
for rule in rule_objects:
    family = rule.get("family", "?")
    table = rule.get("table", "?")
    chain = rule.get("chain", "?")
    handle = rule.get("handle", "unknown")

    rule_id = f"rule:{family}:{table}:{chain}:{handle}"

    G.add_node(
        rule_id,
        kind="rule",
        label=f"{chain}\n#{handle}",
        family=family,
        table=table,
        chain=chain,
    )

    for map_name in extract_map_references(rule):
        target = f"map:{family}:{table}:{map_name}"

        if target not in G:
            G.add_node(
                target,
                kind="unresolved",
                label=f"{family} {table}\n{map_name}",
                family=family,
                table=table,
                name=map_name,
            )

        G.add_edge(rule_id, target, relation="vmap")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected components:", nx.number_weakly_connected_components(G))

In [ ]:
import matplotlib.pyplot as plt

if G.number_of_nodes():
    fig, ax = plt.subplots(figsize=(16, 10))

    pos = nx.spring_layout(G, seed=7, k=1.4)

    colors = []
    for _, attrs in G.nodes(data=True):
        colors.append({
            "rule": "#F6D7A7",
            "map": "#BBD8F5",
            "set": "#C7E8C1",
            "unresolved": "#F2B8B5",
        }.get(attrs.get("kind"), "#DDDDDD"))

    nx.draw_networkx(
        G,
        pos=pos,
        ax=ax,
        node_color=colors,
        node_size=1800,
        font_size=7,
        arrows=True,
        arrowsize=15,
        width=1.2,
    )

    ax.set_title("nftables rule-to-map relationships")
    ax.axis("off")
    plt.show()

In [ ]:
audit_findings = []

# Maps with the same name in multiple locations
locations_by_name = defaultdict(list)

for obj in map_objects:
    locations_by_name[obj.get("name")].append(object_id(obj))

for name, locations in locations_by_name.items():
    if len(locations) > 1:
        audit_findings.append({
            "severity": "INFO",
            "check": "Repeated map name",
            "object": name,
            "details": (
                f"Declared in {len(locations)} locations: "
                f"{locations}"
            ),
        })

# Map objects that have no detected rule references
referenced_identities = {
    row["resolved_identity"]
    for row in reference_audit
    if row["resolution"] == "resolved"
}

for obj in map_objects:
    identity = object_id(obj)

    if identity not in referenced_identities:
        audit_findings.append({
            "severity": "REVIEW",
            "check": "No detected rule reference",
            "object": identity,
            "details": (
                "No reference was resolved by the initial "
                "rule-expression scanner."
            ),
        })

# Unresolved references
for row in reference_audit:
    if row["resolution"] != "resolved":
        audit_findings.append({
            "severity": "REVIEW",
            "check": "Unresolved map reference",
            "object": row["map_name"],
            "details": (
                f"{row['rule_family']} {row['rule_table']} "
                f"{row['rule_chain']} handle "
                f"{row['rule_handle']}: {row['resolution']}"
            ),
        })

audit_df = pd.DataFrame(audit_findings)

if audit_df.empty:
    print("No findings from the initial audit checks.")
else:
    display(
        audit_df.sort_values(
            ["severity", "check"]
        ).reset_index(drop=True)
    )

In [ ]:
def show_objects(kind, limit=3):
    selected = idx.get(kind, [])

    print(f"{kind}: {len(selected)} objects")

    for obj in selected[:limit]:
        display(JSON(obj))


show_objects("map", limit=2)
show_objects("element", limit=2)
show_objects("rule", limit=2)